# Experimentos reproducibles del informe

Esta notebook genera las Tablas **6, 7, 8, 9, 10, 11 y 12** del informe a partir de los modelos de `models/queue_model.py` y `models/simulation.py`. Incluye además el control puntual (1.000 vehículos, semilla 42) descrito en la sección de verificación.

**Importante:** los aforos del proyecto son sintéticos. Los resultados permiten reproducir y verificar los cálculos bajo los supuestos del modelo; no constituyen mediciones del tránsito real de Salta. La simulación M/M/c no reproduce explícitamente el ciclo rojo-verde.

Ejecutá la notebook completa con **Run All** desde la raíz del proyecto o desde `notebooks/`. Se guardará una copia CSV de cada tabla en `results/tables/`.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Localizar la raíz del repositorio para que las rutas funcionen desde root o notebooks/.
candidatas = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
RAIZ_REPO = next(
    (p for p in candidatas if (p / "models" / "queue_model.py").exists()),
    None,
)
if RAIZ_REPO is None:
    raise FileNotFoundError("No encuentro models/queue_model.py; abrí la notebook desde el repositorio.")
if str(RAIZ_REPO) not in sys.path:
    sys.path.insert(0, str(RAIZ_REPO))

from models.queue_model import (
    calcular_mmc,
    calcular_demora_webster,
    calcular_webster,
)
from models.simulation import ejecutar_simulacion

print(f"Raíz del proyecto: {RAIZ_REPO}")


Raíz del proyecto: /workspaces/optimizacion-semaforos-salta


## Parámetros comunes y utilidades

Para las comparaciones se usan 30 réplicas, 20.000 vehículos totales por réplica, 2.000 vehículos descartados como calentamiento y un IC t de Student del 95 % sobre las medias de réplica. La columna `± IC95` representa el **semiancho** del intervalo, no la desviación estándar.


In [2]:
REPLICAS = 30
VEHICULOS_POR_REPLICA = 20_000
WARMUP = 2_000
SEMILLA_BASE = 42

# Parámetros de diseño declarados en el informe.
FLUJO_SATURACION_VEH_H_CARRIL = 1_440.0
FLUJO_SATURACION_VEH_MIN_CARRIL = FLUJO_SATURACION_VEH_H_CARRIL / 60
CICLO_BASE_S = 90.0
TIEMPO_PERDIDO_S = 8.0

CARPETA_TABLAS = RAIZ_REPO / "results" / "tables"
CARPETA_TABLAS.mkdir(parents=True, exist_ok=True)

def guardar_tabla(df, nombre_archivo):
    ruta = CARPETA_TABLAS / nombre_archivo
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"CSV guardado: {ruta}")

def simular_wq(lamda_veh_min, mu_veh_min_por_servidor, carriles, *, seed_base=SEMILLA_BASE):
    """Devuelve la salida común de SimPy para una condición estable."""
    return ejecutar_simulacion(
        lamda=lamda_veh_min,
        mu=mu_veh_min_por_servidor,
        c=carriles,
        num_vehiculos=VEHICULOS_POR_REPLICA,
        warmup=WARMUP,
        replicas=REPLICAS,
        seed_base=seed_base,
    )

def resumen_simulacion(resultado):
    inferior = resultado["IC95_Wq_min"]["inferior"]
    superior = resultado["IC95_Wq_min"]["superior"]
    media = resultado["media_Wq_min"]
    semiancho = (superior - inferior) / 2
    return {
        "Wq_simulado_min": media,
        "IC95_semiancho_min": semiancho,
        "IC95_inferior_min": inferior,
        "IC95_superior_min": superior,
        "vehiculos_simulados": resultado["total_vehiculos_simulados"],
        "vehiculos_medidos": resultado["total_vehiculos_medidos"],
    }

print(f"Configuración de simulación: {REPLICAS} réplicas × {VEHICULOS_POR_REPLICA:,} vehículos; "
      f"warm-up={WARMUP:,}; semilla base={SEMILLA_BASE}.")


Configuración de simulación: 30 réplicas × 20,000 vehículos; warm-up=2,000; semilla base=42.


## Experimento 1 — Tabla 6: solución analítica frente a simulación

Barrido de `λ = 6, 9, 12, 14, 15, 16, 17 veh/min`, con `μ = 6 veh/min` por servidor y `c = 3`. El error relativo se calcula como `(Wq_simulado − Wq_analítico) / Wq_analítico × 100`. El signo indica sobreestimación (+) o subestimación (−).


In [3]:
filas_tabla6 = []
for lamda in [6, 9, 12, 14, 15, 16, 17]:
    analitico = calcular_mmc(lamda=lamda, mu=6.0, c=3)
    sim = simular_wq(lamda, 6.0, 3)
    resumen = resumen_simulacion(sim)
    wq_a = analitico["Wq"]
    wq_s = resumen["Wq_simulado_min"]
    filas_tabla6.append({
        "lambda_veh_min": lamda,
        "rho": analitico["rho"],
        "Wq_analitico_min": wq_a,
        "Wq_simulado_min": wq_s,
        "IC95_semiancho_min": resumen["IC95_semiancho_min"],
        "IC95_inferior_min": resumen["IC95_inferior_min"],
        "IC95_superior_min": resumen["IC95_superior_min"],
        "error_relativo_pct": (wq_s - wq_a) / wq_a * 100,
        "vehiculos_simulados": resumen["vehiculos_simulados"],
        "vehiculos_medidos": resumen["vehiculos_medidos"],
    })

tabla6 = pd.DataFrame(filas_tabla6)
display(tabla6.round(5))
guardar_tabla(tabla6, "tabla_6_validacion_analitico_simpy.csv")


,lambda_veh_min,rho,Wq_analitico_min,Wq_simulado_min,IC95_semiancho_min,IC95_inferior_min,IC95_superior_min,error_relativo_pct,vehiculos_simulados,vehiculos_medidos
0,6,0.33333,0.00758,0.00769,0.00018,0.00750,0.00787,1.44599,600000,540000
1,9,0.50000,0.02632,0.02646,0.00069,0.02577,0.02715,0.54872,600000,540000
2,12,0.66667,0.07407,0.07475,0.00185,0.07291,0.07660,0.91483,600000,540000
3,14,0.77778,0.15285,0.14816,0.00433,0.14383,0.15248,-3.07273,600000,540000
4,15,0.83333,0.23408,0.22844,0.00853,0.21992,0.23697,-2.40978,600000,540000
5,16,0.88889,0.39875,0.39775,0.02357,0.37419,0.42132,-0.25161,600000,540000
6,17,0.94444,0.89686,0.80667,0.06662,0.74005,0.87329,-10.05595,600000,540000


CSV guardado: /workspaces/optimizacion-semaforos-salta/results/tables/tabla_6_validacion_analitico_simpy.csv


### Control puntual documentado en el informe

Esta es una corrida aparte de la Tabla 6: 1.000 vehículos, semilla 42, una réplica y sin warm-up (parámetros compatibles con el control original). No debe confundirse con el promedio de 30 réplicas.


In [4]:
control_analitico = calcular_mmc(lamda=15, mu=6, c=3)
control_simulado = ejecutar_simulacion(
    lamda=15,
    mu=6,
    c=3,
    num_vehiculos=1_000,
    seed=42,
    warmup=0,
    replicas=1,
)
control_wq = control_simulado["Wq_simulado"]
control = pd.DataFrame([{
    "lambda_veh_min": 15,
    "mu_veh_min_servidor": 6,
    "c": 3,
    "vehiculos": control_simulado["total_vehiculos_simulados"],
    "semilla": 42,
    "Wq_analitico_min": control_analitico["Wq"],
    "Wq_simulado_min": control_wq,
    "error_relativo_pct": (control_wq - control_analitico["Wq"]) / control_analitico["Wq"] * 100,
}])
display(control.round(6))
guardar_tabla(control, "control_puntual_1000_vehiculos.csv")


,lambda_veh_min,mu_veh_min_servidor,c,vehiculos,semilla,Wq_analitico_min,Wq_simulado_min,error_relativo_pct
0,15,6,3,1000,42,0.234082,0.233273,-0.345849


CSV guardado: /workspaces/optimizacion-semaforos-salta/results/tables/control_puntual_1000_vehiculos.csv


## Experimento 2 — Tablas 7 y 8: dimensionamiento

La demanda pico de Av. Bicentenario se vuelve a calcular desde `data/raw/aforos_salta_raw.csv`. El archivo es sintético, tal como aclara el informe. Se usa `s = 24 veh/min/carril`, `u = 0,5` y por tanto `μef = 12 veh/min/carril`. En la Tabla 7 el caso de dos carriles se informa analíticamente (ρ está muy cerca de 1); los casos de tres y cuatro carriles se comparan además con simulación.


In [5]:
ruta_aforos = RAIZ_REPO / "data" / "raw" / "aforos_salta_raw.csv"
if not ruta_aforos.exists():
    raise FileNotFoundError(f"No se encontró el dataset: {ruta_aforos}")
aforos = pd.read_csv(ruta_aforos)
seleccion_pico = aforos[
    (aforos["interseccion"] == "Av. Bicentenario")
    & (aforos["periodo"] == "Pico")
]
if seleccion_pico.empty:
    raise ValueError("No hay registros pico de Av. Bicentenario en el CSV.")
DEMANDA_PICO_VEH_H = float(seleccion_pico["demanda_veh_h"].mean())
LAMBDA_PICO = DEMANDA_PICO_VEH_H / 60
print(f"Demanda media pico: {DEMANDA_PICO_VEH_H:.3f} veh/h = {LAMBDA_PICO:.5f} veh/min")


Demanda media pico: 1424.375 veh/h = 23.73958 veh/min


In [6]:
# Tabla 7: comparación por cantidad de carriles; Webster evaluado con ciclo 90 s y u=0,5.
filas_tabla7 = []
for carriles in [2, 3, 4]:
    mu_efectiva = FLUJO_SATURACION_VEH_MIN_CARRIL * 0.5
    analitico = calcular_mmc(LAMBDA_PICO, mu_efectiva, carriles)
    demora = calcular_demora_webster(
        demanda_veh_h=DEMANDA_PICO_VEH_H,
        carriles=carriles,
        flujo_saturacion_veh_h_por_carril=FLUJO_SATURACION_VEH_H_CARRIL,
        ciclo_s=CICLO_BASE_S,
        verde_efectivo_s=CICLO_BASE_S * 0.5,
    )
    fila = {
        "carriles": carriles,
        "rho": analitico["rho"],
        "P_espera": analitico["P_espera"],
        "Lq_veh": analitico["Lq"],
        "Wq_analitico_s": analitico["Wq"] * 60,
        "demora_Webster_s": demora["demora_total_s"],
        "Wq_simulado_s": np.nan,
        "IC95_semiancho_s": np.nan,
        "nota": "Solo analítico: utilización próxima a 1" if carriles == 2 else "",
    }
    if carriles in [3, 4]:
        sim = resumen_simulacion(simular_wq(LAMBDA_PICO, mu_efectiva, carriles))
        fila["Wq_simulado_s"] = sim["Wq_simulado_min"] * 60
        fila["IC95_semiancho_s"] = sim["IC95_semiancho_min"] * 60
    filas_tabla7.append(fila)

tabla7 = pd.DataFrame(filas_tabla7)
display(tabla7.round(4))
guardar_tabla(tabla7, "tabla_7_dimensionamiento_carriles.csv")


,carriles,rho,P_espera,Lq_veh,Wq_analitico_s,demora_Webster_s,Wq_simulado_s,IC95_semiancho_s,nota
0,2,0.9891,0.9838,89.6790,226.6568,136.2085,NaN,NaN,Solo analítico: utilización próxima a 1
1,3,0.6594,0.4343,0.8409,2.1254,18.3975,2.1443,0.0664,
2,4,0.4946,0.1686,0.1650,0.4170,15.5575,0.4294,0.0147,


CSV guardado: /workspaces/optimizacion-semaforos-salta/results/tables/tabla_7_dimensionamiento_carriles.csv


In [7]:
# Tabla 8: c=2 y distintos repartos de verde.
filas_tabla8 = []
for proporcion_verde in [0.5, 0.556, 0.6, 0.667]:
    mu_efectiva = FLUJO_SATURACION_VEH_MIN_CARRIL * proporcion_verde
    analitico = calcular_mmc(LAMBDA_PICO, mu_efectiva, 2)
    filas_tabla8.append({
        "proporcion_verde": proporcion_verde,
        "mu_efectiva_veh_min_carril": mu_efectiva,
        "rho": analitico["rho"],
        "Wq_analitico_s": analitico["Wq"] * 60,
    })

tabla8 = pd.DataFrame(filas_tabla8)
display(tabla8.round(4))
guardar_tabla(tabla8, "tabla_8_proporcion_verde.csv")


,proporcion_verde,mu_efectiva_veh_min_carril,rho,Wq_analitico_s
0,0.500,12.000,0.9891,226.6568
1,0.556,13.344,0.8895,17.0433
2,0.600,14.400,0.8243,8.8321
3,0.667,16.008,0.7415,4.5775


CSV guardado: /workspaces/optimizacion-semaforos-salta/results/tables/tabla_8_proporcion_verde.csv


## Experimento 3 — Tabla 9: sensibilidad a la demanda

Se mantiene `c = 3`, `u = 0,5` y se modifica la demanda de Av. Bicentenario entre −30 % y +40 %. `Wq` se calcula con M/M/c; `d Webster` es la demora de la fórmula de Webster para ciclo de 90 s y verde efectivo de 45 s.


In [8]:
filas_tabla9 = []
for cambio_pct in [-30, -20, -10, 0, 10, 20, 30, 40]:
    lamda = LAMBDA_PICO * (1 + cambio_pct / 100)
    analitico = calcular_mmc(lamda, FLUJO_SATURACION_VEH_MIN_CARRIL * 0.5, 3)
    demora = calcular_demora_webster(
        demanda_veh_h=lamda * 60,
        carriles=3,
        flujo_saturacion_veh_h_por_carril=FLUJO_SATURACION_VEH_H_CARRIL,
        ciclo_s=CICLO_BASE_S,
        verde_efectivo_s=CICLO_BASE_S * 0.5,
    )
    filas_tabla9.append({
        "variacion_demanda_pct": cambio_pct,
        "lambda_veh_min": lamda,
        "rho": analitico["rho"],
        "Lq_veh": analitico["Lq"],
        "Wq_s": analitico["Wq"] * 60,
        "demora_Webster_s": demora["demora_total_s"],
    })

tabla9 = pd.DataFrame(filas_tabla9)
display(tabla9.round(4))
guardar_tabla(tabla9, "tabla_9_sensibilidad_demanda.csv")


,variacion_demanda_pct,lambda_veh_min,rho,Lq_veh,Wq_s,demora_Webster_s
0,-30,16.6177,0.4616,0.1692,0.6109,15.3401
1,-20,18.9917,0.5275,0.2984,0.9426,16.2111
2,-10,21.3656,0.5935,0.5058,1.4204,17.2137
3,0,23.7396,0.6594,0.8409,2.1254,18.3975
4,10,26.1135,0.7254,1.4002,3.2171,19.8534
5,20,28.4875,0.7913,2.4008,5.0566,21.7754
6,30,30.8615,0.8573,4.4609,8.6728,24.6945
7,40,33.2354,0.9232,10.3197,18.6303,30.9136


CSV guardado: /workspaces/optimizacion-semaforos-salta/results/tables/tabla_9_sensibilidad_demanda.csv


## Experimento 4 — Tablas 10 y 11: ciclo y reparto de verde

Se usan los flujos redondeados informados para la intersección hipotética: Bicentenario `q1=1.424 veh/h`, tres carriles; Entre Ríos `q2=1.187 veh/h`, dos carriles; saturación `1.440 veh/h/carril`; tiempo perdido total `tp=8 s`. Para A, el verde efectivo restante se reparte por igual (41 s por fase). Para B se reparte proporcionalmente a `yi`; para C se usa Webster. La demora ponderada se pondera por los flujos de fase.


In [9]:
FASES = [
    {"nombre": "Av. Bicentenario", "demanda_veh_h": 1_424.0, "carriles": 3},
    {"nombre": "Av. Entre Ríos", "demanda_veh_h": 1_187.0, "carriles": 2},
]

def evaluar_configuracion(nombre, ciclo_s, verdes_s):
    resultados = []
    for fase, verde in zip(FASES, verdes_s):
        demora = calcular_demora_webster(
            demanda_veh_h=fase["demanda_veh_h"],
            carriles=fase["carriles"],
            flujo_saturacion_veh_h_por_carril=FLUJO_SATURACION_VEH_H_CARRIL,
            ciclo_s=ciclo_s,
            verde_efectivo_s=verde,
        )
        resultados.append({
            "fase": fase["nombre"],
            "demanda_veh_h": fase["demanda_veh_h"],
            "carriles": fase["carriles"],
            "verde_s": verde,
            "x": demora["x_i"],
            "demora_s": demora["demora_total_s"],
            "proporcion_verde": demora["u_i"],
        })
    total_flujo = sum(f["demanda_veh_h"] for f in FASES)
    demora_ponderada = sum(r["demanda_veh_h"] * r["demora_s"] for r in resultados) / total_flujo
    return {
        "configuracion": nombre,
        "ciclo_s": ciclo_s,
        "fases": resultados,
        "demora_media_ponderada_s": demora_ponderada,
    }

Y_por_fase = [
    f["demanda_veh_h"] / (f["carriles"] * FLUJO_SATURACION_VEH_H_CARRIL)
    for f in FASES
]
Y_TOTAL = sum(Y_por_fase)

config_A = evaluar_configuracion("A: ciclo 90 s, equitativo", 90.0, [41.0, 41.0])
verdes_B = [(90.0 - TIEMPO_PERDIDO_S) * y / Y_TOTAL for y in Y_por_fase]
config_B = evaluar_configuracion("B: ciclo 90 s, proporcional", 90.0, verdes_B)
webster = calcular_webster(
    FASES,
    flujo_saturacion_veh_h_por_carril=FLUJO_SATURACION_VEH_H_CARRIL,
    tiempo_perdido_total_s=TIEMPO_PERDIDO_S,
)
verdes_C = [fase["verde_efectivo_s"] for fase in webster["fases"]]
config_C = evaluar_configuracion("C: Webster", webster["ciclo_webster_s"], verdes_C)

configuraciones = [config_A, config_B, config_C]


In [10]:
filas_tabla10 = []
for config in configuraciones:
    f1, f2 = config["fases"]
    filas_tabla10.append({
        "configuracion": config["configuracion"],
        "ciclo_s": config["ciclo_s"],
        "g1_s": f1["verde_s"],
        "g2_s": f2["verde_s"],
        "x1": f1["x"],
        "x2": f2["x"],
        "demora_fase1_s": f1["demora_s"],
        "demora_fase2_s": f2["demora_s"],
        "demora_media_ponderada_s": config["demora_media_ponderada_s"],
    })

tabla10 = pd.DataFrame(filas_tabla10)
display(tabla10.round(4))
print(f"Webster: Y={webster['Y']:.6f}; ciclo={webster['ciclo_webster_s']:.4f} s")
guardar_tabla(tabla10, "tabla_10_configuraciones_semaforicas.csv")


,configuracion,ciclo_s,g1_s,g2_s,x1,x2,demora_fase1_s,demora_fase2_s,demora_media_ponderada_s
0,"A: ciclo 90 s, equitativo",90.0000,41.0000,41.0000,0.7236,0.9047,22.2920,35.7191,28.3962
1,"B: ciclo 90 s, proporcional",90.0000,36.4388,45.5612,0.8142,0.8142,28.2829,24.0717,26.3684
2,C: Webster,65.8359,25.7009,32.1351,0.8444,0.8444,24.0407,21.6212,22.9408


Webster: Y=0.741782; ciclo=65.8359 s
CSV guardado: /workspaces/optimizacion-semaforos-salta/results/tables/tabla_10_configuraciones_semaforicas.csv


In [11]:
filas_tabla11 = []
for ciclo in [45, 50, 55, 60, 65, 70, 80, 90, 110, 140]:
    if ciclo <= TIEMPO_PERDIDO_S:
        continue
    verdes = [(ciclo - TIEMPO_PERDIDO_S) * y / Y_TOTAL for y in Y_por_fase]
    config = evaluar_configuracion(f"Ciclo {ciclo} s", float(ciclo), verdes)
    filas_tabla11.append({
        "ciclo_s": ciclo,
        "g1_s": config["fases"][0]["verde_s"],
        "g2_s": config["fases"][1]["verde_s"],
        "demora_media_ponderada_s": config["demora_media_ponderada_s"],
    })

tabla11 = pd.DataFrame(filas_tabla11)
display(tabla11.round(4))
guardar_tabla(tabla11, "tabla_11_demora_segun_ciclo.csv")


,ciclo_s,g1_s,g2_s,demora_media_ponderada_s
0,45,16.4419,20.5581,23.9764
1,50,18.6638,23.3362,22.6826
2,55,20.8856,26.1144,22.3466
3,60,23.1075,28.8925,22.4733
4,65,25.3294,31.6706,22.8589
5,70,27.5513,34.4487,23.4039
6,80,31.9950,40.0050,24.7771
7,90,36.4388,45.5612,26.3684
8,110,45.3263,56.6737,29.8747
9,140,58.6575,73.3425,35.4988


CSV guardado: /workspaces/optimizacion-semaforos-salta/results/tables/tabla_11_demora_segun_ciclo.csv


## Tabla 12: verificación M/M/c en las seis fases de las configuraciones A, B y C

Para cada fase se convierte `q` de veh/h a `λ` veh/min. La tasa de servicio por carril es `μef = 24 × (g/C)` veh/min. SimPy estima solo la componente M/M/c `Wq`; no incluye la demora uniforme por semáforo rojo que sí forma parte de la demora Webster de la Tabla 10.


In [12]:
filas_tabla12 = []
for config_id, config in zip(["A", "B", "C"], configuraciones):
    for fase in config["fases"]:
        lamda = fase["demanda_veh_h"] / 60
        mu_efectiva = FLUJO_SATURACION_VEH_MIN_CARRIL * fase["proporcion_verde"]
        analitico = calcular_mmc(lamda, mu_efectiva, fase["carriles"])
        sim = resumen_simulacion(simular_wq(lamda, mu_efectiva, fase["carriles"]))
        wq_a_s = analitico["Wq"] * 60
        wq_s_s = sim["Wq_simulado_min"] * 60
        filas_tabla12.append({
            "configuracion": config_id,
            "fase": fase["fase"],
            "rho": analitico["rho"],
            "Wq_analitico_s": wq_a_s,
            "Wq_simulado_s": wq_s_s,
            "IC95_semiancho_s": sim["IC95_semiancho_min"] * 60,
            "IC95_inferior_s": sim["IC95_inferior_min"] * 60,
            "IC95_superior_s": sim["IC95_superior_min"] * 60,
            "analitico_dentro_IC95": (
                sim["IC95_inferior_min"] <= analitico["Wq"] <= sim["IC95_superior_min"]
            ),
        })

tabla12 = pd.DataFrame(filas_tabla12)
display(tabla12.round(4))
guardar_tabla(tabla12, "tabla_12_verificacion_fases.csv")


,configuracion,fase,rho,Wq_analitico_s,Wq_simulado_s,IC95_semiancho_s,IC95_inferior_s,IC95_superior_s,analitico_dentro_IC95
0,A,Av. Bicentenario,0.7236,3.4901,3.5618,0.0963,3.4655,3.6581,True
1,A,Av. Entre Ríos,0.9047,24.7528,25.1510,1.6602,23.4908,26.8112,True
2,B,Av. Bicentenario,0.8142,7.4242,7.4241,0.2507,7.1734,7.6747,True
3,B,Av. Entre Ríos,0.8142,9.7088,10.0414,0.3900,9.6515,10.4314,True
4,C,Av. Bicentenario,0.8444,9.8887,10.2284,0.6166,9.6118,10.8450,True
5,C,Av. Entre Ríos,0.8444,12.7236,12.7294,0.5842,12.1453,13.3136,True


CSV guardado: /workspaces/optimizacion-semaforos-salta/results/tables/tabla_12_verificacion_fases.csv
